In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    base_url="https://api.openai.com/v1",
    api_key="api")

In [4]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [8]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [9]:
assistant.total_cost()

0.00047175000000000006

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course — is it still okay to join now?',
 'answer_llm': 'Yes — you can still join now.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course — is it still okay to join now?',
 'answer_llm': 'Yes — you can still join now. If you want a certificate, just make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
assistant.reset_usage()

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [16]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/565 [00:00<?, ?it/s]

In [17]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [18]:
assistant.total_cost()

0.6084727500000008

In [20]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)

In [21]:
import pandas as pd

df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [23]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [24]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [26]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [27]:
rec = answers[0]

In [28]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [29]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: joining is still allowed, and certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent.', score='good')

In [30]:
calc_price(usage)

{'input_cost': 0.00021975, 'output_cost': 0.0002385, 'total_cost': 0.00045825}

In [31]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [32]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the course can still be joined now, and that certificate eligibility requires submitting the project while submissions are still open. No key information is missing or altered.', score='good')

In [33]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [34]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/565 [00:00<?, ?it/s]

In [35]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [36]:
df_eval = pd.DataFrame(evaluations)

In [37]:
calc_total_price(usages)

0.3981614999999996

In [38]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 537/565 = 95.04%


In [39]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
1,"Can I start the course late, or is it already ...",74eb249bbf,bad,"The ground truth says late start is allowed, b..."
3,What do I need to do to qualify for the certif...,74eb249bbf,bad,The AI answer is not semantically equivalent t...
28,Do I need to peer-review other students’ capst...,69d122f12e,bad,The AI answer is not semantically equivalent. ...
34,"If homework isn't required, what role does it ...",9f689c185f,bad,The AI answer does not address the question or...
40,When is the next llm-zoomcamp course starting?,bd31146b0e,bad,The ground truth states a specific start time:...


In [40]:
df_eval.to_csv("rag-evaluations-new.csv", index=False)